In [1]:
import argparse
import time
import numpy as np
import wandb

from reasoning.tools.utils import load_model_with_vllm
from datasets import load_dataset
from vllm import SamplingParams
from reasoning.inference.majority import MajorityInference
from reasoning.tools.logger import load_config, apply_config
import numpy as npapi

import argparse
from reasoning.inference.majority import MajorityInference
from reasoning.evaluator.math_grader import math_equal, extract_answer

from reasoning.models.model import ValueModel_qwen


config = load_config("/satassdscratch/byuan48/inference-on-LLMs/configs/experiments_418/development.yaml")
apply_config(config) 
print(config)

# load dataset and model
dataset = load_dataset(config["data_path"])
dataset = dataset['test']
model_name = config["model_name"]

policy_model, tokenizer = load_model_with_vllm(model_name, task='auto', tensor_parallel_size=len(config["cuda_device_ids"]), gpu_memory_utilization=0.8)
tokenizer.pad_token = tokenizer.eos_token 
reward_model = ValueModel_qwen(device = "auto") # default one
start_problem_index = config.get("begin_problem_index", 0)
end_problem_index = config.get("end_problem_index", len(dataset)-1)
dataset = dataset.select(range(start_problem_index, end_problem_index))

# inference hyperparameters
num_return_sequences = config["num_return_sequences"]
assert num_return_sequences == 1
max_new_tokens = config["max_new_tokens"]
temperature = config["temperature"]
top_p = config["top_p"]
system_prompt = config["system_prompt"]

# SamplingTree parameters
sampling_method = config["sampling_method"]
samplingTree_temperature = config["samplingTree_temperature"]
beam_width = config["beam_width"]
max_steps = config["max_steps"]
threshold = config["threshold"]
ORM_type = config["ORM_type"]
# name of the results file
config_name = config["config_name"]

# Also set the seed for the sampling params 
sampling_params = SamplingParams(
    temperature=temperature,
    max_tokens=max_new_tokens,
    n=num_return_sequences,
    top_p=top_p,
    stop_token_ids=[tokenizer.eos_token_id],
    skip_special_tokens = True,
    include_stop_str_in_output = False,
    seed  = config["seed"]
) # shouldn't set seed for random sampling


wandb: Currently logged in as: yuanbo096. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


{'config_name': 'test', 'cuda_device_ids': [0, 1, 2, 3, 4, 5, 6, 7], 'seed': 0, 'data_path': 'HuggingFaceH4/MATH-500', 'model_name': 'meta-llama/Llama-3.2-1B-Instruct', 'reward_model_name': 'Qwen/Qwen2.5-Math-PRM-7B', 'batch_size': 1, 'max_new_tokens': 2048, 'temperature': 0.7, 'top_p': 0.9, 'sampling_method': 'stochastic_beam_search_version4', 'num_return_sequences': 1, 'samplingTree_temperature': 0.1, 'beam_width': 4, 'max_steps': 2, 'threshold': 0.9, 'wandb_project': 'efficient_reasoning', 'ORM_type': 'product', 'system_prompt': 'Solve the following math problem efficiently and clearly:\n\n- For simple problems (2 steps or fewer):\n  Provide a concise solution with minimal explanation.\n\n- For complex problems (3 steps or more):\n  Use this step-by-step format:\n\n  ## Step 1: [Concise description]\n  [Brief explanation and calculations]\n\n  ## Step 2: [Concise description]\n  [Brief explanation and calculations]\n\n  ...\n\n  Regardless of the approach, always conclude with:\n\n 

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

WARNING 04-19 13:26:33 multiproc_worker_utils.py:312] Reducing Torch parallelism from 24 threads to 1 to avoid unnecessary CPU contention. Set OMP_NUM_THREADS in the external environment to tune this value as needed.
INFO 04-19 13:26:33 custom_cache_manager.py:17] Setting Triton cache manager to: vllm.triton_utils.custom_cache_manager:CustomCacheManager
INFO 04-19 13:26:35 selector.py:217] Cannot use FlashAttention-2 backend for Volta and Turing GPUs.
INFO 04-19 13:26:35 selector.py:129] Using XFormers backend.
(VllmWorkerProcess pid=207923) INFO 04-19 13:26:35 selector.py:217] Cannot use FlashAttention-2 backend for Volta and Turing GPUs.
(VllmWorkerProcess pid=207923) INFO 04-19 13:26:35 selector.py:129] Using XFormers backend.
(VllmWorkerProcess pid=207926) INFO 04-19 13:26:35 selector.py:217] Cannot use FlashAttention-2 backend for Volta and Turing GPUs.
(VllmWorkerProcess pid=207924) (VllmWorkerProcess pid=207926) INFO 04-19 13:26:35 selector.py:217] Cannot use FlashAttention-2 ba

ValueError: Bfloat16 is only supported on GPUs with compute capability of at least 8.0. Your NVIDIA GeForce RTX 2080 Ti GPU has compute capability 7.5. You can use float16 instead by explicitly setting the`dtype` flag in CLI, for example: --dtype=half.

(VllmWorkerProcess pid=207927) INFO 04-19 13:26:39 multiproc_worker_utils.py:222] Worker ready; awaiting tasks
(VllmWorkerProcess pid=207923) ERROR 04-19 13:26:39 multiproc_worker_utils.py:236] Exception in worker VllmWorkerProcess while processing method init_device.
(VllmWorkerProcess pid=207923) ERROR 04-19 13:26:39 multiproc_worker_utils.py:236] Traceback (most recent call last):
(VllmWorkerProcess pid=207923) ERROR 04-19 13:26:39 multiproc_worker_utils.py:236]   File "/satassdscratch/byuan48/software/anaconda3/envs/reasoning_transformers/lib/python3.12/site-packages/vllm/executor/multiproc_worker_utils.py", line 230, in _run_worker_process
(VllmWorkerProcess pid=207923) ERROR 04-19 13:26:39 multiproc_worker_utils.py:236]     output = executor(*args, **kwargs)
(VllmWorkerProcess pid=207923) ERROR 04-19 13:26:39 multiproc_worker_utils.py:236]              ^^^^^^^^^^^^^^^^^^^^^^^^^
(VllmWorkerProcess pid=207923) (VllmWorkerProcess pid=207924) ERROR 04-19 13:26:39 multiproc_worker_uti

In [ ]:
import importlib
import reasoning.inference.tree
tree_module = importlib.reload(reasoning.inference.tree)
Tree = tree_module.Tree

In [ ]:

start_time = time.time()
right_count = 0
num_generated_tokens = 0
error_steps_count_all = 0
print(f'length of the dataset: {len(dataset)}')
for i in range(0, len(dataset)):
    question = dataset['problem'][i]
    answer = dataset['solution'][i]        

    tree = Tree(system_prompt, question, policy_model, reward_model, sampling_method, config_name, sampling_params, samplingTree_temperature, beam_width, threshold, ORM_type)
    for _ in range(max_steps):
        path = tree.generate_next_trajectory() 
        tree.paths.append(path)
    majority_inference = MajorityInference(policy_model=policy_model, tokenizer=tokenizer, sampling_params=sampling_params, config_name=config_name, reward_model=reward_model, method = 'weighted_majority', ORM_type=ORM_type)
    all_solutions  = [path.solutions for path in tree.explored_paths]
    rewards = [tree.get_reward(path.scores) for path in tree.explored_paths]
    
    solution = majority_inference.weighted_majority_vote(all_solutions, rewards)   
    extracted_answer = extract_answer(answer)
    is_correct = math_equal(extracted_answer, solution)
    if extracted_answer is None:
        raise ValueError('extracted answer is None')
    if is_correct:
        right_count += 1
    num_generated_tokens += tree.num_generated_tokens
    print("--------------------------------")
    print(f'question: {question}')
    print(f'solution: {solution}')
    print(f'extracted answer: {extracted_answer}')
    print(f'is correct: {is_correct}')
    print(f'Accuracy: {right_count/(i+1)}')
    wandb.log({"Accuracy": right_count/(i+1)}, step=i)
    print(f'Number of generated tokens: {tree.num_generated_tokens}')
    # print out the error steps count
    error_steps_count = 0
    for path in tree.explored_paths:
        if len(path.steps) <= 1:
            error_steps_count += 1
    error_steps_count_all += error_steps_count
    print(f'Error steps count: {error_steps_count_all}')
    wandb.log({"Error steps count": error_steps_count_all}, step=i)
    print("--------------------------------")


wandb.log({"Accuracy": right_count/len(dataset)})
wandb.log({"Number of generated tokens": num_generated_tokens/len(dataset)})
end_time = time.time()
print("Time taken: {} seconds".format(end_time - start_time))

wandb.finish()